In [0]:
# run_all.py

from src.endpoints import list_keys_all, list_keys_changes, next_offset
from src.connection import APIClient
from src.loaders import fetch_all_keys_to_bronze, fetch_items_to_silver_json
from src.curate_items import kuratoi_ja_talleta_deltaan_like_batch
from utils.gpcutils import print_unique_gpc_codes_and_names
from src.config import *
import time
import pandas as pd  

# ----------------------------
# 0) Valitse ajotapa
# ----------------------------
# "full"    = koko katalogi (overwrite)
# "changes" = vain muutokset (append)
RUN_MODE = "full"

# ----------------------------
# 1) Asetukset
# ----------------------------
BASE = "https://productapi-synkka.gs1.fi"
EMAIL = dbutils.secrets.get("gs1-kv", "email")
PASSWORD = dbutils.secrets.get("gs1-kv", "password")
GLN = dbutils.secrets.get("gs1-kv", "gln")

access_key = dbutils.secrets.get("gs1-kv", "storage-access-key")
spark.conf.set(f"fs.azure.account.key.{ACCOUNT}.dfs.core.windows.net", access_key)


# (suositus, auttaa skeemamuutoksiin append-ajossa)
if SPARK_DELTA_AUTOMERGE:
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# Parametrit
KEY_BATCH_SIZE = 90000
ITEM_BATCH     = 1000                            # max 1000 / Many

# ----------------------------
# 2) Autentikointi
# ----------------------------
client = APIClient(BASE, EMAIL, PASSWORD, GLN)
client.authenticate(verbose=True)

# ----------------------------
# 3) Ajohaarojen logiikka
# ----------------------------
if RUN_MODE == "full":
    # --- 3A: KOKO KANTA ---
    print(">>> FULL-ajo: hae kaikki avaimet ja kaikki tuotteet")

    # All → Bronze
    all_keys = fetch_all_keys_to_bronze(spark, client, DELTA_KEYS, batch_size=KEY_BATCH_SIZE)
    print(f"Avaimien (All) nouto onnistui, avaimia talletettu: {all_keys}")

    # Bronze → Silver (kaikki Id:t) — overwrite
    n_fetched = fetch_items_to_silver_json(
        spark,
        client,
        silver_path=DELTA_PRODUCTS,
        bronze_path=DELTA_KEYS,
        batch_size=ITEM_BATCH,
        first_write_mode="overwrite",
        rate_limit_per_minute=RATE_LIMIT_PER_MIN,
        verbose=True,
        only_gpc_segment_code=ONLY_GPC_SEGMENT_CODE,   # <-- suodatus käyttöön
    )
    print(f"Koko kannan tuotteet haettu ja tallennettu Silveriin. Rivimäärä: {n_fetched}")

else:
    # --- 3B: VAIN MUUTOKSET ---
    print(f">>> CHANGES-ajo: hae muutokset alkaen {SINCE_ISO.split('T', 1)[0]}")

    chg_ids = []
    offset = None
    while True:
        resp_chg = list_keys_changes(client, batch_size=KEY_BATCH_SIZE, since=SINCE_ISO, offset=offset)
        items = resp_chg.get("Items") or []
        got = [it.get("Id") for it in items if it.get("Id")]
        chg_ids.extend(got)

        print(f"Haettu sivu, uusia ID:itä: {len(got)}, yhteensä: {len(chg_ids)}")

        offset = next_offset(resp_chg)
        if not offset:
            break
        time.sleep(1.0)  # pieni tauko sivutuksen väliin

    print(f"Avaimien (Changes) nouto onnistui, avaimia haettu yhteensä: {len(chg_ids)}")

    if chg_ids:
        # Id-lista → Silver (vain nämä) — append
        n_fetched = fetch_items_to_silver_json(
            spark,
            client,
            silver_path=DELTA_PRODUCTS,
            ids=chg_ids,
            batch_size=ITEM_BATCH,
            first_write_mode="overwrite",
            rate_limit_per_minute=RATE_LIMIT_PER_MIN,
            verbose=True,
            only_gpc_segment_code=ONLY_GPC_SEGMENT_CODE,  # <-- suodatus käyttöön
        )
        print(f"Muuttuneiden tuotteiden tiedot noudettu ja tallennettu Silveriin. Rivimäärä: {n_fetched}")
    else:
        print("Ei muutoksia haettavaksi.")

# ----------------------------
# 4) Pikavilkaisu Silveriin (Pandas)
# ----------------------------
silver_df = spark.read.format("delta").load(DELTA_PRODUCTS)
print("Silver-taulun skeema:")
silver_df.printSchema()

print("Esimerkkirivejä Silveristä (Pandas):")
cols = [c for c in ["Id", "ingest_ts", "raw_json"] if c in silver_df.columns]
pdf_silver = silver_df.select(*cols).limit(5).toPandas()

# lyhennetään raw_json esikatseluun
if "raw_json" in pdf_silver.columns:
    pd.set_option("display.max_colwidth", 200)
    pdf_silver["raw_json"] = pdf_silver["raw_json"].str.slice(0, 200)

print(pdf_silver)

#--------------------------------------------------------
# 4.5) GPC-koodit & nimet Silveristä (diagnostiikka/print)
#--------------------------------------------------------------
print_unique_gpc_codes_and_names(spark, DELTA_PRODUCTS)


# ----------------------------
# 5) Kuratoi → Curated
# ----------------------------
print(">>> Vaihe: Kuratoi tuotteet ja tallenna Deltaan")
#curate_mode = "overwrite" if RUN_MODE == "full" else "append"
curate_mode = "overwrite"

rows = kuratoi_ja_talleta_deltaan_like_batch(
    spark,
    silver_path=DELTA_PRODUCTS,
    curated_path=CURATED_ITEMS,
    write_mode=curate_mode,
    sample_rows=5
)

print(f"Kuratoituja rivejä kirjoitettu: {rows}")

# ----------------------------
# 6) Pikavilkaisu Curatediin (Pandas)
# ----------------------------
curated_df = spark.read.format("delta").load(CURATED_ITEMS)
cols_curated = ["Id", "BrandName", "TradeItemDescription_fi", "GTIN", "PrimaryImageUrl"]
cols_curated = [c for c in cols_curated if c in curated_df.columns]

print("Esimerkkirivejä Curatedista (Pandas):")
pdf_curated = curated_df.select(*cols_curated).limit(5).toPandas()
print(pdf_curated)


In [0]:
from utils.azuresqlserver import write_append, read_table
from src.config import *

curated_df = spark.read.format("delta").load(CURATED_ITEMS)

# kirjoitus
write_append(curated_df, SQL_TABLE_CURATED_ITEMS, dbutils)

# pikacheck ilman sarakeluetteloa
sample = read_table(
    spark,
    SQL_TABLE_CURATED_ITEMS,
    dbutils,
    top_n=5,          # hae 5 ensimmäistä
    to_pandas=True,   # pandasiksi tulostusta varten
)
print(sample)


In [0]:
# run_all.py

from src.config import *
from src.add_kesko_hierarchy_levels.enrich_kesko_categories import enrich_curated_with_kesko_categories

# 0) (valinnainen sanity) – näet että polku tulee configista oikein
print("OUT PATH:", CURATED_ITEMS_WITH_KESKO)

access_key = dbutils.secrets.get("gs1-kv", "storage-access-key")
spark.conf.set(f"fs.azure.account.key.{ACCOUNT}.dfs.core.windows.net", access_key)

# 3) rikastus – polku tulee suoraan CONFIGISTA (ei subpatheja)
res = enrich_curated_with_kesko_categories(
    spark,
    dbutils,
    output_path=CURATED_ITEMS_WITH_KESKO,   # ABFSS-URI configista
    curated_table=SQL_TABLE_CURATED_ITEMS,  # tai None -> sama oletus
    kesko_levels_table="dbo.KESKO_00_PRODUCT_HIERARCHY_LEVELS",
    write_mode="overwrite",
)

print(res)


In [0]:

from utils.azuresqlserver import write_append, read_table
from src.config import SQL_TABLE_KESKO_CATEGORIES, CURATED_ITEMS_WITH_KESKO, ACCOUNT

access_key = dbutils.secrets.get("gs1-kv", "storage-access-key")
spark.conf.set(f"fs.azure.account.key.{ACCOUNT}.dfs.core.windows.net", access_key)

curated_df = spark.read.format("delta").load(CURATED_ITEMS_WITH_KESKO)

# kirjoitus
write_append(curated_df, SQL_TABLE_KESKO_CATEGORIES, dbutils)

# pikacheck ilman sarakeluetteloa
sample = read_table(
    spark,
    SQL_TABLE_KESKO_CATEGORIES,
    dbutils,
    top_n=5,          # hae 5 ensimmäistä
    to_pandas=True,   # pandasiksi tulostusta varten
)
print(sample)